In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Frames on the wire: camera nodes and clients

Component 1 gave every model a home. Component 2 gives the *data* a home:
ROS 2 topics. The architecture rule is simple and non-negotiable:

> **Server nodes own hardware. Everything else subscribes.**

One node owns the mount camera; one node owns the robot bus. Behaviors,
previews, loggers, and your debugging terminal all watch the same topics.
Any piece can be restarted, replaced, or inspected (`ros2 topic echo`)
without touching the rest.

![Alt Text](../../animations/Camera_Node_Animation.gif)

**Matching TODOs:** `2.1`-`2.3` in `project/ros/camera_node.py` and
`project/ros/camera_client.py`.

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import time
import threading

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
from common.ros_helpers import to_image_msg  # ndarray->Image (provided codec)
from common.ros_helpers import from_image_msg

cfg = load_config()
print("running against:", PROJECT_ROOT)

In [ ]:
# Inline image display helper (BGR -> notebook).


def show(img_bgr, title="", figsize=(9, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# ROS 2 check: this notebook runs on real ROS 2 (rclpy) only.
try:
    import rclpy

    HAVE_ROS = True
except ImportError:
    HAVE_ROS = False

ROS_HELP = """rclpy is not importable in this kernel, so the ROS 2 cells cannot run."""


def require_ros():
    if not HAVE_ROS:
        raise RuntimeError(ROS_HELP)


if HAVE_ROS:
    if not rclpy.ok():
        rclpy.init()
    print("real ROS 2 (rclpy) ready")
else:
    print(ROS_HELP)

> **About background nodes.** The demo nodes this notebook creates are
> self-contained: they publish their own data, so the lesson works with
> nothing else running. To also run the *real* device nodes (the SO-101
> arm, the cameras, the intent node), start them yourself in a terminal;
> the "On the rig" section below has the exact commands. Every subscriber
> simply reads whatever is on the topic, so both can coexist.

## 1. Quality of Service (QoS)

Every camera topic in this pipeline uses:

```python
QoSProfile(reliability=ReliabilityPolicy.BEST_EFFORT,
           history=HistoryPolicy.KEEP_LAST, depth=1)
```

- **best-effort**: a lost frame is *free*; the next one is 33 ms behind.
  Reliable delivery would make DDS re-transmit stale pixels while fresh
  ones queue behind them.
- **keep-last, depth 1**: the latest frame always wins. Vision must never
  back up behind a slow consumer; there is no value in frame #4823 once
  #4824 exists.

And the trap every ROS beginner hits once: **reliability must be
compatible on both ends.** A RELIABLE *subscriber* demands guarantees a
BEST_EFFORT *publisher* never offers, so DDS simply refuses to match them.
No error. No warning. Just silence. We'll reproduce that live in section 4.

## 2. A minimal camera node

This is the demo twin of your TODO 2.1/2.2: a node that owns the real
mount camera and publishes `Image` + `Bool`. Read every line; the project
version adds config plumbing but no new ideas.

In [ ]:
require_ros()

from rclpy.node import Node
from rclpy.qos import HistoryPolicy, QoSProfile, ReliabilityPolicy
from sensor_msgs.msg import Image
from std_msgs.msg import Bool

SENSOR_QOS = QoSProfile(
    reliability=ReliabilityPolicy.BEST_EFFORT, history=HistoryPolicy.KEEP_LAST, depth=1
)


def try_open_mount_camera():
    """The real mount camera: the configured V4L device, else the default
    webcam. Returns ``(capture or None, device tried)`` - no exceptions, so
    the node can keep retrying it while running."""
    device = cfg["cameras"]["mount"].get("device", 0)
    candidates = [device, 0]
    if isinstance(device, str) and not pathlib.Path(device).exists():
        candidates = [0]  # missing path: skip it quietly (probing
        # it makes OpenCV warn noisily)
    for candidate in candidates:
        cap = (
            cv2.VideoCapture(candidate, cv2.CAP_V4L2)  # V4L2 direct: no
            if isinstance(candidate, str)  # CV_IMAGES probing
            else cv2.VideoCapture(candidate)
        )
        if cap.isOpened():
            return cap, candidate
        cap.release()
    return None, device


class DemoCameraNode(Node):
    """Owns the mount camera - and honors the graceful-failure contract:
    unplug the camera and the node STAYS UP, publishes ``healthy: false``,
    and keeps retrying the device until it comes back. Behaviors degrade
    instead of dying."""

    REOPEN_EVERY_S = 2.0

    def __init__(self, topic="/cameras/mount/image_raw", fps=15.0):
        super().__init__("demo_camera_node")
        self.source, which = try_open_mount_camera()
        if self.source is None:
            raise RuntimeError(
                "could not open the mount camera. To fix:\n"
                f"  - check it is plugged in (config expects {which})\n"
                "  - close any other app holding the camera\n"
                "then re-run this cell."
            )
        print(f"camera open: {which}")
        self._closing = False
        self._next_reopen = 0.0
        self.pub = self.create_publisher(Image, topic, SENSOR_QOS)
        self.health_pub = self.create_publisher(
            Bool, "/cameras/mount/healthy", SENSOR_QOS
        )
        self._timer = self.create_timer(1.0 / fps, self.tick)  # the heartbeat

    def tick(self):
        if self._closing:  # shutting down: go quiet
            return
        ok, frame = False, None
        if self.source is not None:
            ok, frame = self.source.read()
            if not ok:  # unplugged mid-stream?
                self.source.release()
                self.source = None
                self._next_reopen = time.time() + self.REOPEN_EVERY_S
                self.get_logger().warning(
                    "camera read failed - node stays up, healthy=False, retrying"
                )
        elif time.time() >= self._next_reopen:  # periodic reopen attempt
            self.source, which = try_open_mount_camera()
            self._next_reopen = time.time() + self.REOPEN_EVERY_S
            if self.source is not None:
                self.get_logger().info(f"camera back: {which}")
        try:
            if ok:
                self.pub.publish(
                    to_image_msg(frame, node=self, frame_id="camera_mount")
                )
            self.health_pub.publish(Bool(data=ok))  # health rides along, always
        except Exception:
            pass  # torn down mid-publish: ignore

    def close(self):
        """Silence the heartbeat FIRST, then give the camera back."""
        self._closing = True
        try:
            self._timer.cancel()
        except Exception:
            pass
        if self.source is not None:
            self.source.release()
            self.source = None


cam_node = DemoCameraNode()
print("node created:", cam_node.get_name())

In [ ]:
require_ros()

# A tiny helper to spin nodes on background executors, so the notebook
# stays interactive while nodes tick. (This mirrors how the pipeline's
# orchestrator embeds its client nodes.)


class Runner:
    def __init__(self, node):
        from rclpy.executors import SingleThreadedExecutor

        self.node, self.executor = node, SingleThreadedExecutor()
        self.executor.add_node(node)
        threading.Thread(target=self.executor.spin, daemon=True).start()

    def stop(self):
        self.executor.shutdown()
        try:
            self.node.destroy_node()
        except Exception:
            pass


def wait_for(pred, timeout_s=3.0, interval=0.02):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        v = pred()
        if v:
            return v
        time.sleep(interval)
    return None

In [ ]:
# Spin the node on a background executor, the notebook's stand-in for
# "runs in its own terminal". (On the rig, real device nodes are separate
# processes; see section 5 for the commands.)
cam_runner = Runner(cam_node)
print("camera node ticking in the background")

## 3. The client side: subscribe, cache the newest, tolerate staleness

The consumer pattern (your TODO 2.3): subscribe with matching QoS, let a
background executor service callbacks, keep only the latest frame, and
treat a stale stream as a dead camera. Behaviors then just call
`read() -> (ok, frame)` and never know ROS is underneath.

In [ ]:
require_ros()


class DemoCameraClient:
    def __init__(self, topic="/cameras/mount/image_raw", stale_after_s=1.0):
        from rclpy.executors import SingleThreadedExecutor

        self._lock, self._frame, self._stamp = threading.Lock(), None, 0.0
        self._stale = stale_after_s
        self.node = rclpy.create_node("demo_camera_client")
        self.node.create_subscription(Image, topic, self._on_image, SENSOR_QOS)
        self._exec = SingleThreadedExecutor()
        self._exec.add_node(self.node)
        threading.Thread(target=self._exec.spin, daemon=True).start()

    def _on_image(self, msg):
        frame = from_image_msg(msg)  # Image->ndarray (provided)
        with self._lock:
            self._frame, self._stamp = frame, time.time()

    def read(self):
        with self._lock:
            if self._frame is None or time.time() - self._stamp > self._stale:
                return False, None
            return True, self._frame


client = DemoCameraClient()
wait_for(lambda: client.read()[0], 3.0)
ok, frame = client.read()
print("frame received over the topic:", ok, frame.shape if ok else None)
show(frame, "received via /cameras/mount/image_raw")

That one frame crossed a real pub/sub boundary. Now watch it live: the
loop below keeps pulling the newest frame from the client and redraws it in
place until you press **stop**, then you move on. This is the whole
consumer pattern working at camera rate: node captures, topic carries,
client caches, you read.

While it runs, try unplugging the camera: the picture freezes, the node
stays up publishing `healthy: false` and retrying, and the moment you plug
it back in the stream resumes. That is the graceful-failure contract from
the real node's spec, working in the demo node too.

In [ ]:
require_ros()


def to_jpeg(img):
    ok, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return jpg.tobytes()


stop_btn = widgets.Button(description="stop", button_style="danger")
stopped = {"flag": False}
stop_btn.on_click(lambda _: stopped.update(flag=True))

# ONE persistent Image widget, updated in place. Re-displaying a fresh
# image every frame tears the old DOM node down first - that is what made
# the view blink. Swapping the bytes on the same widget is flicker-free.
view = widgets.Image(format="jpeg")
status = widgets.Label("waiting for the first frame...")
display(widgets.VBox([stop_btn, view, status]))

shown = 0
with ui_events() as poll:
    while not stopped["flag"]:
        poll(10)  # let the stop click arrive
        ok, live = client.read()
        if ok:
            view.value = to_jpeg(live)
            shown += 1
            status.value = f"live - {shown} frames received over the topic"
        else:
            status.value = (
                "no fresh frame (stale or unhealthy) - "
                "watching /cameras/mount/healthy tells you which"
            )
        time.sleep(0.03)
print(f"stopped after {shown} frames received over the topic")

That image crossed a real pub/sub boundary: serialized to a
`sensor_msgs/Image` (raw `bgr8` bytes + width/height/step), delivered over
DDS by topic name, decoded back. On the rig the exact same path carries
your webcam at 30 fps.

## 4. The QoS-mismatch trap, live

Same topic. One change: the subscriber asks for RELIABLE. Watch it hear
*nothing*:

In [ ]:
require_ros()

received = []
strict_qos = QoSProfile(
    reliability=ReliabilityPolicy.RELIABLE,  # <- the mistake
    history=HistoryPolicy.KEEP_LAST,
    depth=1,
)
strict_node = rclpy.create_node("strict_subscriber")
strict_node.create_subscription(
    Image, "/cameras/mount/image_raw", lambda m: received.append(m), strict_qos
)
strict_runner = Runner(strict_node)

time.sleep(1.5)
print(f"frames heard by the RELIABLE subscriber in 1.5 s: {len(received)}")
print("(the best-effort publisher can never satisfy a reliable subscriber;")
print(" DDS refuses the match SILENTLY. Diagnose it with:")
print("   ros2 topic info /cameras/mount/image_raw -v )")
strict_runner.stop()

Remember this one. It is the number-one "my subscriber gets nothing" cause
in real deployments, and the self-test harness checks your client for
exactly this mistake.

## 5. On the rig: real processes, real tools

In real operation each node is a separate process. After you implement the
TODOs, that's:

```bash
# Terminal 1 - the node owns the device (synthetic works anywhere):
python -m ros.camera_node --role mount [--synthetic]

# Terminal 2 - inspect the stream like an operator:
ros2 node list
ros2 topic hz  /cameras/mount/image_raw     # about the configured fps
ros2 topic echo /cameras/mount/healthy      # True / False heartbeats
ros2 topic info /cameras/mount/image_raw -v # QoS on every endpoint

# Terminal 3 - your client, as a live preview:
python -m ros.camera_client --role mount
```

Graceful failure is part of the contract: unplug the camera and the real
node *stays up*, publishes `healthy: false`, and retries. Behaviors degrade
instead of dying.

In [ ]:
# housekeeping - order matters. Silence the camera node's timer FIRST and
# only then stop executors and destroy nodes: a timer tick firing
# mid-teardown would publish on a destroyed publisher and raise
# InvalidHandle.
cam_node.close()  # stop the heartbeat + release the camera
cam_runner.stop()  # stop its executor, destroy the node
client._exec.shutdown()  # stop the client's executor
try:
    client.node.destroy_node()
except Exception:
    pass
print("demo nodes stopped, camera released")

## 6. Your turn

Open **`project/ros/camera_node.py`** and **`project/ros/camera_client.py`**:

| TODO | What | Taught in |
|---|---|---|
| **2.1** | `CameraNode._setup_ros`: publishers + timer | section 2 |
| **2.2** | `CameraNode._publish`: Image + health | section 2 |
| **2.3** | `Ros2Camera._setup_ros`: subscription + executor | sections 3 and 4 |

Then, from the `project/` directory:

```bash
python -m ros.selftest        # round-trips your nodes over synthetic frames

# to see all the TODOs you have left
python app.py
```

The harness saves the received frame to `_artifacts/camera_roundtrip.png`
and, if your round-trip goes silent, prints the publisher/subscriber graph
and any QoS mismatch it saw, by name.

Next: `02_robot.ipynb`, joints on the wire.